In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
pd.set_option('display.max_rows', None)

#Load data
htrain = pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/train.csv")
htest  = pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/test.csv")

In [2]:
def preprocess(df):
    df = df.copy()

    # TotalSF has the 3rd greatest effect on XGB and is very right skewed
    # logging TotalSF greatly improves the RMSLE
    df['TotalSF'] = np.log1p(df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF'])
    df['HomeAge'] = df['YrSold'] - df['YearBuilt']

    # Drop columns
    # Prefer GarageArea over GarageCars because SF allows more granularity
    df.drop(labels=[
        'Fence','PoolQC','Alley','Street',
        'GrLivArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF',
        'TotalBsmtSF','2ndFlrSF','LowQualFinSF','1stFlrSF',
        'GarageCars','TotRmsAbvGrd','GarageYrBlt',
        'YrSold','YearBuilt'
    ], axis=1, inplace=True)

    # One-hot encode nominal categoricals with no natural order
    df = pd.get_dummies(df, columns=['MSZoning','LotConfig','Neighborhood',
                                      'RoofMatl','RoofStyle'])
    df = pd.get_dummies(df, columns=[
        'Exterior1st','Exterior2nd','MasVnrType','Foundation','Heating',
        'GarageType','MiscFeature','SaleType','SaleCondition',
        'CentralAir','Electrical','Functional','PavedDrive'
    ])

    # Ordinal encode categoricals with a natural order
    # Mapped by average SalePrice where no obvious order exists
    ordinal_maps = {
        'LotShape':    {'Reg':1,'IR0':2,'IR2':3,'IR1':4},
        'LandContour': {'Bnk':1,'Lvl':2,'Low':3,'HLS':4},
        'Utilities':   {'AllPub':1,'NoSeWa':2},
        'LandSlope':   {'Gtl':1,'Mod':2,'Sev':3},
        'Condition1':  {'Artery':1,'RRAe':2,'Feedr':3,'RRAn':4,'Norm':5,'RRNe':6,'RRNn':7,'PosN':8,'PosA':9},
        'Condition2':  {'RRNn':1,'Artery':2,'Feedr':3,'RRAn':4,'Norm':5,'RRAe':6,'PosN':7,'PosA':8},
        'BldgType':    {'2fmCon':1,'Duplex':2,'Twnhs':3,'TwnhsE':4,'1Fam':5},
        'HouseStyle':  {'SFoyer':1,'SLvl':2,'1.5Unf':3,'2.5Unf':4,'1.5Fin':5,'1Story':6,'2Story':7,'2.5Fin':8},
        'GarageCond':  {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        'GarageFinish':{'Unf':1,'RFn':2,'Fin':3},
        'GarageQual':  {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        'FireplaceQu': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        'BsmtExposure':{'No':1,'Mn':2,'Av':3,'Gd':4},
        'KitchenQual': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        'HeatingQC':   {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        'BsmtFinType1':{'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
        'BsmtFinType2':{'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
        'BsmtCond':    {'Po':1,'Fa':2,'TA':3,'Gd':4},
        'BsmtQual':    {'Po':1,'Fa':2,'TA':3,'Gd':4},
        'ExterCond':   {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
        # ExterQual is the most powerful regressor of all
        'ExterQual':   {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    }
    for col, mapping in ordinal_maps.items():
        df[f'{col}_enc'] = df[col].map(mapping)
        df.drop(columns=col, inplace=True)

    df = df.fillna(0)
    return df


In [4]:
#Preprocess both train and test 
htrain_processed = preprocess(htrain)
htest_processed  = preprocess(htest)

# Align columns test may be missing OHE columns that train has
X_full = htrain_processed.drop(columns='SalePrice')
htest_processed = htest_processed.reindex(columns=X_full.columns, fill_value=0)


# Train/test split for evaluation
y = np.log1p(htrain_processed['SalePrice'])
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, random_state=104, test_size=0.25, shuffle=True)


# Models
xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.03,
                          max_depth=3, subsample=0.8,
                          colsample_bytree=0.8, random_state=1)
xgb_model.fit(X_train, y_train)

lgbm_model = LGBMRegressor(n_estimators=300, learning_rate=0.05,
                            num_leaves=31, random_state=1)
lgbm_model.fit(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lasso_model = Lasso(alpha=0.001, random_state=1)
lasso_model.fit(X_train_scaled, y_train)


#Evaluate on held-out test set
ensemble_pred = (0.3 * xgb_model.predict(X_test) +
                 0.1 * lgbm_model.predict(X_test) +
                 0.6 * lasso_model.predict(X_test_scaled))

rmsle = np.sqrt(np.mean((y_test - ensemble_pred) ** 2))
r2    = r2_score(y_test, ensemble_pred)
print(f"Ensemble RMSLE: {rmsle:.4f}")
print(f"R²:             {r2:.4f}")


# Retrain on full data and submit 
y_full = np.log1p(htrain_processed['SalePrice'])

xgb_model.fit(X_full, y_full)
lgbm_model.fit(X_full, y_full)

scaler_full   = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)
htest_scaled  = scaler_full.transform(htest_processed)

lasso_model.fit(X_full_scaled, y_full)

#By trial and error I found 0.3,0.1 and 0.6 to be the most accurate
test_preds = (0.3 * xgb_model.predict(htest_processed) +
              0.1 * lgbm_model.predict(htest_processed) +
              0.6 * lasso_model.predict(htest_scaled))

# Reverse the log transform to get back to dollar scale
final_predictions = np.expm1(test_preds)

submission = pd.DataFrame({
    'Id': htest['Id'],
    'SalePrice': final_predictions
})
submission.to_csv('submission.csv', index=False)
print(submission.head())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000754 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2033
[LightGBM] [Info] Number of data points in the train set: 1095, number of used features: 114
[LightGBM] [Info] Start training from score 12.028095
Ensemble RMSLE: 0.1165
R²:             0.9215
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000817 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2273
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 122
[LightGBM] [Info] Start training f

mappingEC= {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
htrain['ExterCond_enc'] = htrain['ExterCond'].map(mappingEC)
htrain.drop(columns='ExterCond', inplace=True)Score is based on Root Mean Square Error